# 1 - Imports

In [24]:
%reload_ext autoreload
%autoreload 2

In [25]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0].parents[0]))

In [ ]:
from src.utils import config, io
from src.preprocessing import preprocess_pipeline
from src.models import model_pipeline, evaluate

# 2 - Preprocessing

In [27]:
X = io.load_csv(config.PROCESSED_DATA_DIR / 'X.csv', index_col=0)
y = io.load_csv(config.PROCESSED_DATA_DIR / 'y.csv', index_col=0)

In [28]:
split_cfg = io.load_json(config.PROCESSED_DATA_DIR / 'splits/temporal_v1.json')
split_cfg

{'description': 'Forecasting split for future country risk prediction',
 'train_years': [1999, 2015],
 'val_years': [2016, 2018],
 'test_years': [2019, 2024]}

In [29]:
def get_subset_data(data, bounds):
    return data[(data['YEAR'] >= bounds[0]) & (data['YEAR'] <= bounds[1])]

In [30]:
X_train = get_subset_data(X, split_cfg['train_years'])
y_train = y.loc[X_train.index]
X_val = get_subset_data(X, split_cfg['val_years'])
y_val = y.loc[X_val.index]
X_test = get_subset_data(X, split_cfg['test_years'])
y_test = y.loc[X_test.index]

In [33]:
model_name = 'logistic_regression'

In [32]:
import mlflow

mlflow.set_tracking_uri(config.PROJECT_ROOT / 'models/mlruns')
mlflow.set_experiment('Country Risk Prediction')


<Experiment: artifact_location='file:///Users/hippolytegrandet/Desktop/Dev/country_risk_rating/models/mlruns/991689756472581023', creation_time=1770566631957, experiment_id='991689756472581023', last_update_time=1770566631957, lifecycle_stage='active', name='Country Risk Prediction', tags={}>

# 3 - Parameter Tuning Logistic Regression Model

In [ ]:
from src.features import selection, pruning

In [111]:
def get_dataset_feature_selected(dataset, params):
    non_float_features_t = list(dataset.columns[dataset.dtypes!=float])

    print('Initial Dataset Shape:', dataset.shape)
    dataset = dataset[dataset['OECD_RATING'] != '-']
    # print('Drop Null Target Shape:', dataset.shape)
    dataset = selection.filter_missingness(
        dataset, 
        max_missing_ratio=params['max_missing_ratio']
    )
    # print('Filter Missingness Shape:', dataset.shape)
    dataset = dataset[non_float_features_t].join(selection.filter_low_variance(
        dataset.drop(columns=non_float_features_t), 
        threshold=params['low_var_threshold']
    ))
    # print('Filter Low Variance Shape:', dataset.shape)
    dataset = dataset[non_float_features_t].join(selection.filter_correlated(
        dataset.drop(columns=non_float_features_t),
        max_corr=params['max_corr']
    ))
    # print('Filter Correlated Shape:', dataset.shape)
    dataset = dataset[non_float_features_t].join(selection.select_by_mutual_information(
        dataset.drop(columns=non_float_features_t),
        dataset['OECD_RATING'],
        top_k=params['top_k_mi']
    ))
    # print('Filter Mutual Information Shape:', dataset.shape)
    # print('Final Dataset Shape:', dataset.shape)
    return dataset

In [ ]:
def get_X_y(dataset):
    # Drop Instances with little data
    dataset = dataset[(dataset.isna().sum(axis=1) / len(dataset.columns)) <= 0.5]

    non_float_features_t = list(dataset.columns[dataset.dtypes!=float])
    non_float_features = []
    for c in non_float_features_t:
        if c != 'OECD_RATING':
            non_float_features.append(c)

    X = dataset.drop(columns=['OECD_RATING']) # Drop Target
    X = X.drop(columns=['ISO3_COUNTRY_CODE']) # Drop Country ID (maybe YEAR)
    y = dataset['OECD_RATING']

    return X, y

def transform_initial_dataset(dataset, feature_selection_params):

    dataset = get_dataset_feature_selected(dataset, feature_selection_params)
    X, y = get_X_y(dataset, feature_selection_params)

    return X, y

In [ ]:
def split_dataset(X, y, split_cfg):
    X_train = get_subset_data(X, split_cfg['train_years'])
    y_train = y.loc[X_train.index]
    X_val = get_subset_data(X, split_cfg['val_years'])
    y_val = y.loc[X_val.index]
    X_test = get_subset_data(X, split_cfg['test_years'])
    y_test = y.loc[X_test.index]

    return X_train, y_train, X_val, y_val, X_test, y_test


## 3.1 - Parameter Tuning Feature Selection

In [ ]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'merged_dataset.csv', index_col=0)
dataset

In [49]:
feature_selection_grid = {
    'max_missing_ratio': [0.2, 0.3, 0.4, 0.5, 0.6],
    'low_var_threshold': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'max_corr': [0.7, 0.75, 0.8, 0.85, 0.9, 0.95],
    'top_k_mi': [25, 50, 100, 200]
}

In [ ]:
from itertools import product

for max_missing_ratio, low_var_threshold, max_corr, top_k_mi in product(
    feature_selection_grid['max_missing_ratio'], feature_selection_grid['low_var_threshold'], feature_selection_grid['max_corr'], feature_selection_grid['top_k_mi'], 
):

    X, y = transform_initial_dataset(dataset, {
        'max_missing_ratio': max_missing_ratio, 'low_var_threshold': low_var_threshold, 'max_corr': max_corr, 'top_k_mi': top_k_mi
    })

    X_train, X_test

In [ ]:
non_float_features_t = list(dataset.columns[dataset.dtypes!=float])

print('Initial Dataset Shape:', dataset.shape)
dataset = dataset[dataset['OECD_RATING'] != '-']
# print('Drop Null Target Shape:', dataset.shape)
dataset = selection.filter_missingness(
    dataset, 
    max_missing_ratio=feature_selection_grid['max_missing_ratio']
)
# print('Filter Missingness Shape:', dataset.shape)
dataset = dataset[non_float_features_t].join(selection.filter_low_variance(
    dataset.drop(columns=non_float_features_t), 
    threshold=feature_selection_grid['low_var_threshold']
))
# print('Filter Low Variance Shape:', dataset.shape)
dataset = dataset[non_float_features_t].join(selection.filter_correlated(
    dataset.drop(columns=non_float_features_t),
    max_corr=feature_selection_grid['max_corr']
))
# print('Filter Correlated Shape:', dataset.shape)
dataset = dataset[non_float_features_t].join(selection.select_by_mutual_information(
    dataset.drop(columns=non_float_features_t),
    dataset['OECD_RATING'],
    top_k=feature_selection_grid['top_k_mi']
))
# print('Filter Mutual Information Shape:', dataset.shape)
# print('Final Dataset Shape:', dataset.shape)
dataset

## 3.2 - Parameter Tuning Preprocessor

In [52]:
preprocessor_params_grid

{'num_imputer': {'knn': {'weights': ['uniform', 'distance'],
   'n_neighbors': [3, 5, 7, 11]},
  'uni': {'strategy': ['mean', 'median', 'most_frequent', 'constant'],
   'fill_value': 0.0}},
 'num_scaler': True,
 'cat_imputer': {'uni': {'strategy': ['most_frequent', 'constant'],
   'fill_value': 'NaN'}}}

In [97]:
s = [
    [1, 2],
    ['a', 'b']
]

In [98]:
product(s)

NameError: name 'product' is not defined

In [105]:
def get_list_params(k, e, prev_param):
    new_l = []
    if type(e) == dict:
        l = []
        for k2, e2 in e.items():
            print('K, V Pairing:', k2, e2)
            print('Result for IT:', get_list_params(k2, e2, prev_param))
            print('FOR', k, k2)

            v = get_list_params(k2, e2, prev_param)

            v_l = []
            if type(v) == dict:
                for v, d in v.items():
                    v_l.append({v: d})
            else:
                v_l = v

            print(k2, e2)
            for a in v_l:
                for b in l:
                    print['A', a]
                    print['B', b]
                    new_l.append(a | b)

        return new_l
    else:
        return e

In [110]:
a = preprocessor_params_grid['num_imputer']['knn']
a

{'weights': ['uniform', 'distance'], 'n_neighbors': [3, 5, 7, 11]}

In [ ]:
ds = []
for k, v in a.items():
    

In [106]:

preprocessor_params_l = []
for k1, e1 in preprocessor_params_grid.items():
    preprocessor_params_l += get_list_params(k1, e1, {})    

K, V Pairing: knn {'weights': ['uniform', 'distance'], 'n_neighbors': [3, 5, 7, 11]}
K, V Pairing: weights ['uniform', 'distance']
Result for IT: ['uniform', 'distance']
FOR knn weights
weights ['uniform', 'distance']
K, V Pairing: n_neighbors [3, 5, 7, 11]
Result for IT: [3, 5, 7, 11]
FOR knn n_neighbors
n_neighbors [3, 5, 7, 11]
Result for IT: []
FOR num_imputer knn
K, V Pairing: weights ['uniform', 'distance']
Result for IT: ['uniform', 'distance']
FOR knn weights
weights ['uniform', 'distance']
K, V Pairing: n_neighbors [3, 5, 7, 11]
Result for IT: [3, 5, 7, 11]
FOR knn n_neighbors
n_neighbors [3, 5, 7, 11]
knn {'weights': ['uniform', 'distance'], 'n_neighbors': [3, 5, 7, 11]}
K, V Pairing: uni {'strategy': ['mean', 'median', 'most_frequent', 'constant'], 'fill_value': [0.0]}
K, V Pairing: strategy ['mean', 'median', 'most_frequent', 'constant']
Result for IT: ['mean', 'median', 'most_frequent', 'constant']
FOR uni strategy
strategy ['mean', 'median', 'most_frequent', 'constant']
K

In [79]:
preprocessor_params_l

['knn', 'uni', {'num_scaler': True}, {'num_scaler': False}, 'cat_imputer', '']

In [68]:
preprocessor_params_grid = {
    # Numeric Transformer
    # Numeric Imputation
    'num_imputer': {
        'knn': {
            'weights': ['uniform', 'distance'],
            'n_neighbors': [3, 5, 7, 11]
        }, 'uni': {
            'strategy': ['mean', 'median', 'most_frequent', 'constant'],
            'fill_value': [0.0]
        }
    },
    # Scalarization
    'num_scaler': [True, False],

    # String Transformer
    'cat': {
        'cat_imputer': {
            'strategy': ['most_frequent', 'constant'],
            'fill_value': ['NaN']
        },
        '': {}
    }
}

In [ ]:
preprocessor = preprocess_pipeline.build_preprocessor(
    X,
    params_grid=preprocessor_params_grid
)

In [ ]:
model_params = {
    'C': 1.0,
    'max_iter': 1000,
    'class_weight': 'balanced'
}

model = model_pipeline.get_model_pipeline(
    model_name,
    preprocessor,
    model_params
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_preprocessor_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median']
}

search_cv = RandomizedSearchCV(model, param_grid, n_iter=10)

In [ ]:
with mlflow.start_run(run_name='baseline_lr_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    val_metrics = evaluate.evaluate_model(model, X_val, y_val, prefix='val_')
    test_metrics = evaluate.evaluate_model(model, X_test, y_test, prefix='test_')

    mlflow.log_metrics({**val_metrics, **test_metrics})

    # Log model
    # mlflow.sklearn.log_model(
    #     model,
    #     artifact_path='model',
    #     registered_model_name=None,
    #     input_example=X_test.loc[[X_test.index[0]]]
    # )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

2026/02/08 17:19:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/utils/validat

Baseline Logistic Regression Results
val_accuracy: 0.6839
val_precision: 0.6272
val_recall: 0.6372
val_f1: 0.6255

Classification Report
              precision    recall  f1-score   support

           1       0.93      0.69      0.79       162
           2       0.67      0.58      0.62        24
           3       0.55      0.72      0.62        46
           4       0.46      0.42      0.44        31
           5       0.45      0.60      0.52        40
           6       0.56      0.65      0.60        94
           7       0.76      0.80      0.78       125

    accuracy                           0.68       522
   macro avg       0.63      0.64      0.63       522
weighted avg       0.71      0.68      0.69       522


Confusion Matrix
[[112   1   2   4   1  21  21]
 [  2  14   8   0   0   0   0]
 [  1   6  33   6   0   0   0]
 [  1   0  13  13   4   0   0]
 [  0   0   4   1  24  10   1]
 [  0   0   0   1  23  61   9]
 [  4   0   0   3   1  17 100]]


2026/02/08 17:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Baseline Logistic Regression Results
test_accuracy: 0.6090
test_precision: 0.5080
test_recall: 0.4956
test_f1: 0.4931

Classification Report
              precision    recall  f1-score   support

           1       0.78      0.70      0.74       270
           2       0.50      0.33      0.40        33
           3       0.53      0.64      0.58        73
           4       0.12      0.07      0.08        46
           5       0.44      0.36      0.39        84
           6       0.44      0.67      0.53       139
           7       0.74      0.70      0.72       217

    accuracy                           0.61       862
   macro avg       0.51      0.50      0.49       862
weighted avg       0.62      0.61      0.61       862


Confusion Matrix
[[189   1   2   8   0  33  37]
 [ 11  11  11   0   0   0   0]
 [ 13   6  47   2   2   3   0]
 [  6   1  15   3  15   6   0]
 [  4   2   7   8  30  32   1]
 [  6   1   3   3  17  93  16]
 [ 12   0   3   1   4  45 152]]


/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in

MLflow run_id: 19e7ce582e774588a933156a2aadfff2
